In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ========================
# ==== CONFIG ============
# ========================
results_path = "ExperimentosSP/results_by_state_year/model_topk_2_norm_minmax_noise_True_ep_50_lr_0.001"

all_data = []

# ========================
# ==== LOAD & MERGE ======
# ========================
for root, _, files in os.walk(results_path):
    subfolder = os.path.basename(root)
    
    for file in files:
        if not file.endswith(".csv"):
            continue

        file_path = os.path.join(root, file)
        df = pd.read_csv(file_path)

        # Exemplo esperado:
        # qualquer_coisa_STATE_YEAR.csv
        parts = file.replace(".csv", "").split("_")
        year = int(parts[-1])
        state = parts[-2].upper()

        df_long = df.melt(
            id_vars=["Modelo"],
            var_name="Produto",
            value_name="MAPE"
        )

        df_long["Year"] = year
        df_long["State"] = state
        df_long["horizonte"] = subfolder 

        all_data.append(df_long)

# ========================
# ==== CLEANUP ===========
# ========================
df_all = pd.concat(all_data, ignore_index=True)

df_all = df_all[df_all["Produto"] != "Unnamed: 0"]

df_all = (
    df_all
    .replace([float("inf"), float("-inf")], pd.NA)
    .dropna(subset=["MAPE"])
)


In [ ]:
target_products = [
    "Etanolhidratado",
    "Gasolinac",
    "Glp",
    "Oleodiesel",
    "Querosenedeaviacao"
]

df_all = df_all[df_all["Produto"].isin(target_products)]

print(df_all.head())


In [ ]:
# ================================================
#  IMPORTS
# ================================================
import numpy as np
import pandas as pd
from itertools import combinations
import seaborn as sns
import matplotlib.pyplot as plt


# ================================================
#  1. BAYESIAN SIGNED-RANK TEST
# ================================================
def bayesian_signed_rank(differences, rope=0.5, s=0.5):
    """
    Implementa o Bayesian Signed-Rank Test.
    differences: array das diferenças (B - A) ou (A - B)
    rope: região de equivalência (ex: 0.5 = 0.5% MAPE)
    s: pseudo-contagem do prior Dirichlet (0.5 recomendado)
    """
    diffs = np.array(differences)

    n_left = np.sum(diffs < -rope)         # B melhor
    n_rope = np.sum(np.abs(diffs) <= rope) # equivalentes
    n_right = np.sum(diffs > rope)         # A melhor

    alpha = np.array([n_left + s, n_rope + s, n_right + s])

    # 5000 amostras do posterior Dirichlet
    posterior = np.random.dirichlet(alpha, size=5000)

    theta_left  = posterior[:, 0].mean()
    theta_rope  = posterior[:, 1].mean()
    theta_right = posterior[:, 2].mean()

    return theta_left, theta_rope, theta_right


# ================================================
#  2. GERAR MATRIZ BAYESIANA
# ================================================
def bayesian_matrix(df, rope=0.5, metric="MAPE"):
    modelos = sorted(df["Modelo"].unique())

    mat_better = pd.DataFrame(0.0, index=modelos, columns=modelos)
    mat_rope   = pd.DataFrame(0.0, index=modelos, columns=modelos)

    for A, B in combinations(modelos, 2):
        dfA = df[df["Modelo"] == A][metric].values
        dfB = df[df["Modelo"] == B][metric].values

        if len(dfA) != len(dfB):
            raise ValueError(f"Modelos {A} e {B} têm tamanhos diferentes após filtragem.")

        # menor = melhor (como MAPE), por isso diffs = B - A
        diffs = dfB - dfA

        theta_left, theta_rope, theta_right = bayesian_signed_rank(diffs, rope)

        # chances do modelo A ser melhor (erro menor)
        mat_better.loc[A, B] = theta_right
        mat_better.loc[B, A] = theta_left

        # prob equivalência
        mat_rope.loc[A, B] = theta_rope
        mat_rope.loc[B, A] = theta_rope

    return mat_better, mat_rope


# ================================================
#  3. PLOT ESTILO EXATAMENTE IGUAL AO DA IMAGEM
# ================================================
def plot_bayesian_matrix(matrix, title=""):
    plt.figure(figsize=(10, 8))

    ax = sns.heatmap(
        matrix,
        annot=True,
        fmt=".3f",
        cmap="Blues",
        vmin=0,
        vmax=1,
        linewidths=0.5,
        linecolor='white',
        cbar=True
    )

    ax.set_title(title, fontsize=16, pad=20)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()


# ================================================
#  4. EXECUTAR TODAS AS COMBINAÇÕES PEDIDAS
# ================================================
def run_bayesian_tests_by_horizonte_and_global(df_all, rope=0.5, metric="MAPE"):
    resultados = {}

    # A) POR HORIZONTE
    for horizonte in df_all["horizonte"].unique():
        sub = df_all[df_all["horizonte"] == horizonte]

        if len(sub["Modelo"].unique()) > 1:
            mat_better, mat_rope = bayesian_matrix(sub, rope, metric)
            resultados[("HORIZONTE", horizonte)] = {
                "matrix_better": mat_better,
                "matrix_rope": mat_rope
            }

    # B) GLOBAL (todos os horizontes juntos)
    if len(df_all["Modelo"].unique()) > 1:
        mat_better, mat_rope = bayesian_matrix(df_all, rope, metric)
        resultados[("GLOBAL", "ALL_HORIZONTES")] = {
            "matrix_better": mat_better,
            "matrix_rope": mat_rope
        }

    return resultados



# =============================================================
#  5. RODAR TUDO E GERAR TODOS OS PLOTS AUTOMATICAMENTE
# =============================================================
rope = 0.5

print("Calculando matrizes bayesianas...")
resultados = run_bayesian_tests_by_horizonte_and_global(
    df_all, rope=rope, metric="MAPE"
)

print("Plotando matrizes...\n")

for key, res in resultados.items():
    tipo, valor = key

    if tipo == "GLOBAL":
        title = f"GLOBAL — Probabilidade de A ser melhor que B (rope = {rope}%)"
    else:
        title = f"Horizonte = {valor} — Probabilidade de A ser melhor que B (rope = {rope}%)"

    plot_bayesian_matrix(res["matrix_better"], title)

In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from itertools import combinations

# ========================
# ==== CONFIG ============
# ========================
BASE_RESULTS_PATH = "ExperimentosSP/results_by_state_year"

target_products = [
    "Etanolhidratado",
    "Gasolinac",
    "Glp",
    "Oleodiesel",
    "Querosenedeaviacao"
]

# ========================
# ==== BAYESIAN TEST =====
# ========================
def bayesian_signed_rank(differences, rope=0.5, s=0.5):
    diffs = np.array(differences)

    n_left  = np.sum(diffs < -rope)
    n_rope  = np.sum(np.abs(diffs) <= rope)
    n_right = np.sum(diffs > rope)

    alpha = np.array([n_left + s, n_rope + s, n_right + s])
    posterior = np.random.dirichlet(alpha, size=5000)

    return posterior.mean(axis=0)


def bayesian_matrix(df, rope=0.5, metric="MAPE"):
    modelos = sorted(df["Modelo"].unique())

    mat = pd.DataFrame(0.0, index=modelos, columns=modelos)

    for A, B in combinations(modelos, 2):
        dfA = df[df["Modelo"] == A][metric].values
        dfB = df[df["Modelo"] == B][metric].values

        if len(dfA) != len(dfB):
            raise ValueError(f"{A} e {B} têm tamanhos diferentes.")

        diffs = dfB - dfA
        _, _, theta_A_better = bayesian_signed_rank(diffs, rope)

        mat.loc[A, B] = theta_A_better
        mat.loc[B, A] = 1 - theta_A_better

    return mat


def plot_bayesian_matrix(matrix, title):
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        matrix,
        annot=True,
        fmt=".3f",
        cmap="Blues",
        vmin=0,
        vmax=1,
        linewidths=0.5,
        linecolor="white",
        cbar=True
    )
    plt.title(title, fontsize=16, pad=20)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

# ========================
# ==== LOOP EXPERIMENTOS =
# ========================
rope = 0.5

for experiment_name in sorted(os.listdir(BASE_RESULTS_PATH)):
    experiment_path = os.path.join(BASE_RESULTS_PATH, experiment_name)

    if not os.path.isdir(experiment_path):
        continue

    print("\n======================================")
    print(f"Experimento: {experiment_name}")
    print("======================================")

    all_data = []

    for root, _, files in os.walk(experiment_path):
        for file in files:
            if not file.endswith(".csv"):
                continue

            file_path = os.path.join(root, file)
            df = pd.read_csv(file_path)

            parts = file.replace(".csv", "").split("_")
            year = int(parts[-1])
            state = parts[-2].upper()

            df_long = df.melt(
                id_vars=["Modelo"],
                var_name="Produto",
                value_name="MAPE"
            )

            df_long["Year"] = year
            df_long["State"] = state

            all_data.append(df_long)

    if not all_data:
        print("⚠️ Nenhum CSV encontrado.")
        continue

    df_all = pd.concat(all_data, ignore_index=True)

    df_all = (
        df_all
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=["MAPE"])
    )

    df_all = df_all[df_all["Produto"].isin(target_products)]

    if df_all["Modelo"].nunique() < 2:
        print("⚠️ Menos de dois modelos — pulando.")
        continue

    mat_global = bayesian_matrix(df_all, rope=rope)

    plot_bayesian_matrix(
        mat_global,
        title=f"{experiment_name} — GLOBAL (rope = {rope}%)"
    )


In [ ]:
import os
import numpy as np
import pandas as pd

# ========================
# CONFIG
# ========================
BASE_RESULTS_PATH = "ExperimentosSP/results_by_state_year"
MY_MOE_NAME = "My-MoE"
ROPE = 0.5

target_products = [
    "Etanolhidratado",
    "Gasolinac",
    "Glp",
    "Oleodiesel",
    "Querosenedeaviacao"
]

# ========================
# BAYESIAN FUNCTIONS
# ========================
def bayesian_signed_rank(differences, rope=0.5, s=0.5):
    diffs = np.array(differences)

    n_left  = np.sum(diffs < -rope)
    n_rope  = np.sum(np.abs(diffs) <= rope)
    n_right = np.sum(diffs > rope)

    alpha = np.array([n_left + s, n_rope + s, n_right + s])
    posterior = np.random.dirichlet(alpha, size=5000)

    return posterior.mean(axis=0)


def prob_A_better_than_B(df, A, B, rope=0.5, metric="MAPE"):
    dfA = df[df["Modelo"] == A][metric].values
    dfB = df[df["Modelo"] == B][metric].values

    if len(dfA) != len(dfB):
        raise ValueError(f"{A} e {B} com tamanhos diferentes.")

    diffs = dfB - dfA
    _, _, theta_A_better = bayesian_signed_rank(diffs, rope)

    return theta_A_better


# ========================
# BUILD GLOBAL DF
# ========================
rows = []

for experiment_name in sorted(os.listdir(BASE_RESULTS_PATH)):
    experiment_path = os.path.join(BASE_RESULTS_PATH, experiment_name)

    if not os.path.isdir(experiment_path):
        continue

    all_data = []

    for root, _, files in os.walk(experiment_path):
        for file in files:
            if not file.endswith(".csv"):
                continue

            df = pd.read_csv(os.path.join(root, file))

            parts = file.replace(".csv", "").split("_")
            year = int(parts[-1])
            state = parts[-2].upper()

            df_long = df.melt(
                id_vars=["Modelo"],
                var_name="Produto",
                value_name="MAPE"
            )

            df_long["Year"] = year
            df_long["State"] = state
            all_data.append(df_long)

    if not all_data:
        continue

    df_all = pd.concat(all_data, ignore_index=True)

    df_all = (
        df_all
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=["MAPE"])
    )

    df_all = df_all[df_all["Produto"].isin(target_products)]

    modelos = df_all["Modelo"].unique()
    my_moe_models = [m for m in modelos if MY_MOE_NAME in m]

    if not my_moe_models:
        continue

    my_moe = my_moe_models[0]

    row = {"Experimento": experiment_name}

    probs = []

    for other_model in modelos:
        if other_model == my_moe:
            continue

        p = prob_A_better_than_B(df_all, my_moe, other_model, rope=ROPE)
        row[f"Prob_MyMoE_vs_{other_model}"] = p
        probs.append(p)

    # score agregado para ordenação
    row["Score_Medio_MyMoE"] = np.mean(probs)

    rows.append(row)

# ========================
# FINAL DATAFRAME
# ========================
df_global = pd.DataFrame(rows).sort_values(
    "Score_Medio_MyMoE", ascending=False
).reset_index(drop=True)

TOP_K = 10

# ========================
# PRINTS PEDIDOS
# ========================
print("\n" + "=" * 80)
print("TOP EXPERIMENTOS — My-MoE MAIS GANHA")
print("=" * 80)
print(df_global.head(TOP_K).to_string(index=False))

print("\n" + "=" * 80)
print("TOP EXPERIMENTOS — My-MoE MAIS PERDE")
print("=" * 80)
print(df_global.tail(TOP_K).to_string(index=False))


In [ ]:
import pandas as pd
from pathlib import Path
import json

# ========================
# CONFIGURAÇÃO
# ========================
BASE_OUTPUT_PATH = "ExperimentosSP/output_dir"

# Diretório para salvar as análises
OUTPUT_ANALYSIS_DIR = "analises_resultados"
Path(OUTPUT_ANALYSIS_DIR).mkdir(exist_ok=True)

# ========================
# FUNÇÕES AUXILIARES
# ========================

def get_base_experiment_name(experiment_name):
    """
    Remove o ano e sufixo _log para obter o nome base
    Ex: model_topk_1_norm_minmax_noise_False_ep_20_lr_0.0001_2020_log
    -> model_topk_1_norm_minmax_noise_False_ep_20_lr_0.0001
    """
    # Remove _log
    if experiment_name.endswith('_log'):
        experiment_name = experiment_name[:-4]
    
    # Remove ano (4 dígitos entre 2000-2030)
    parts = experiment_name.split('_')
    cleaned_parts = []
    for part in parts:
        if len(part) == 4 and part.isdigit():
            year = int(part)
            if 2000 <= year <= 2030:
                continue
        cleaned_parts.append(part)
    
    return '_'.join(cleaned_parts)


def extract_year_from_experiment(experiment_name):
    """
    Extrai o ano do nome do experimento
    """
    parts = experiment_name.split('_')
    for part in parts:
        if len(part) == 4 and part.isdigit():
            year = int(part)
            if 2000 <= year <= 2030:
                return year
    return None


def find_experiment_directory(base_name, base_path):
    """
    Encontra o diretório do experimento baseado no nome base
    """
    base_path = Path(base_path)
    
    # Tenta encontrar diretamente
    possible_names = [
        base_name,
        base_name.replace('model_', ''),
        base_name.replace('model_', '').replace('_', '-'),
    ]
    
    for exp_dir in base_path.iterdir():
        if not exp_dir.is_dir():
            continue
        
        dir_name = exp_dir.name
        
        # Verifica se o nome do diretório corresponde
        for possible_name in possible_names:
            if possible_name in dir_name or dir_name in possible_name:
                return exp_dir
    
    return None


def get_experiment_data_from_directory(base_name, base_path):
    """
    Busca os dados do experimento diretamente na pasta dele
    """
    # Encontra o diretório do experimento
    exp_dir = find_experiment_directory(base_name, base_path)
    
    if exp_dir is None:
        print(f"⚠️ Diretório não encontrado para: {base_name}")
        return None, None
    
    # Lê os arquivos
    epochs_file = exp_dir / "epochs_per_experiment.csv"
    experts_file = exp_dir / "experts_weights_summary.csv"
    
    df_epochs = None
    df_experts = None
    
    if epochs_file.exists():
        df_epochs = pd.read_csv(epochs_file)
    else:
        print(f"⚠️ Arquivo epochs_per_experiment.csv não encontrado em {exp_dir}")
    
    if experts_file.exists():
        df_experts = pd.read_csv(experts_file)
    else:
        print(f"⚠️ Arquivo experts_weights_summary.csv não encontrado em {exp_dir}")
    
    return df_epochs, df_experts


def get_experiment_details_by_base_name(base_name, base_path):
    """
    Retorna detalhes do experimento agrupados por horizonte
    Busca dados diretamente na pasta do experimento
    """
    df_epochs, df_experts = get_experiment_data_from_directory(base_name, base_path)
    
    if df_epochs is None:
        return {}
    
    details = {}
    
    # Agrupa por horizonte
    for horizonte in sorted(df_epochs['horizon'].unique()):
        horizon_data = df_epochs[df_epochs['horizon'] == horizonte]
        
        if horizon_data.empty:
            continue
        
        # Encontra maior e menor número de épocas
        max_epochs_row = horizon_data.loc[horizon_data['num_epochs'].idxmax()]
        min_epochs_row = horizon_data.loc[horizon_data['num_epochs'].idxmin()]
        
        max_epochs = int(max_epochs_row['num_epochs'])
        min_epochs = int(min_epochs_row['num_epochs'])
        
        max_year = int(max_epochs_row['year']) if 'year' in max_epochs_row else extract_year_from_experiment(max_epochs_row['experiment'])
        min_year = int(min_epochs_row['year']) if 'year' in min_epochs_row else extract_year_from_experiment(min_epochs_row['experiment'])
        
        # Pega a min_loss diretamente da linha (que é a loss da última época)
        max_last_loss = float(max_epochs_row['min_loss']) if 'min_loss' in max_epochs_row else None
        min_last_loss = float(min_epochs_row['min_loss']) if 'min_loss' in min_epochs_row else None
        
        # Média de épocas
        media_epochs = horizon_data['num_epochs'].mean()
        
        details[horizonte] = {
            'max_epochs': max_epochs,
            'max_last_loss': max_last_loss,
            'max_year': max_year,
            'min_epochs': min_epochs,
            'min_last_loss': min_last_loss,
            'min_year': min_year,
            'media_epochs': media_epochs
        }
    
    return details


def get_learner_stats_for_base_name(base_name, base_path):
    """
    Retorna estatísticas agregadas de learners
    Busca dados diretamente na pasta do experimento
    """
    _, df_experts = get_experiment_data_from_directory(base_name, base_path)
    
    if df_experts is None or df_experts.empty:
        return pd.DataFrame()
    
    # Agrupa por learner (soma seleções, média de pesos)
    learner_stats = df_experts.groupby('Learner').agg({
        'selections': 'sum',      # Soma total de seleções
        'avg_weight': 'mean'      # Média dos pesos
    }).reset_index()
    
    learner_stats.columns = ['Learner', 'Total Seleções', 'Peso Médio']
    learner_stats['Total Seleções'] = learner_stats['Total Seleções'].astype(int)
    learner_stats = learner_stats.sort_values('Total Seleções', ascending=False)
    
    return learner_stats


# ========================
# PARTE 1: INFO GERAL DE TODOS OS EXPERIMENTOS
# ========================

print("\n" + "=" * 80)
print("INFO GERAL DE TODOS OS EXPERIMENTOS")
print("=" * 80)

epochs_data = []
experts_data = []

base_path = Path(BASE_OUTPUT_PATH)

# Percorre todos os diretórios de experimentos
for exp_dir in base_path.iterdir():
    if not exp_dir.is_dir():
        continue
    
    # Arquivo de épocas
    epochs_file = exp_dir / "epochs_per_experiment.csv"
    if epochs_file.exists():
        df_epochs = pd.read_csv(epochs_file)
        epochs_data.append(df_epochs)
    
    # Arquivo de experts
    experts_file = exp_dir / "experts_weights_summary.csv"
    if experts_file.exists():
        df_experts = pd.read_csv(experts_file)
        experts_data.append(df_experts)

# Concatena todos os dados
all_epochs_df = pd.concat(epochs_data, ignore_index=True)
all_experts_df = pd.concat(experts_data, ignore_index=True)

print(f"\nTotal de experimentos encontrados: {len(epochs_data)}")
print(f"Total de horizontes: {len(all_epochs_df['horizon'].unique())}")
print(f"Horizontes: {sorted(all_epochs_df['horizon'].unique())}")

# Dicionário para armazenar análises gerais
analise_geral = {
    'total_experimentos': len(epochs_data),
    'total_horizontes': len(all_epochs_df['horizon'].unique()),
    'horizontes': sorted(all_epochs_df['horizon'].unique()),
    'analise_por_horizonte': {}
}

# Análise por horizonte
horizontes = all_epochs_df['horizon'].unique()

for horizonte in sorted(horizontes):
    print("\n" + "=" * 80)
    print(f"HORIZONTE: {horizonte}")
    print("=" * 80)
    
    df_horizonte = all_epochs_df[all_epochs_df['horizon'] == horizonte]
    
    # MENOR NÚMERO DE ÉPOCAS
    min_epochs = df_horizonte['num_epochs'].min()
    exp_min_epochs = df_horizonte[df_horizonte['num_epochs'] == min_epochs]
    total_min = len(exp_min_epochs)
    
    print(f"\nMENOR NÚMERO DE ÉPOCAS = {min_epochs}")
    print(f"TOTAL = {total_min}")
    for idx, row in exp_min_epochs.iterrows():
        print(f"  {row['experiment']}")
    
    # MAIOR NÚMERO DE ÉPOCAS
    max_epochs = df_horizonte['num_epochs'].max()
    exp_max_epochs = df_horizonte[df_horizonte['num_epochs'] == max_epochs]
    total_max = len(exp_max_epochs)
    
    print(f"\nMAIOR NÚMERO DE ÉPOCAS = {max_epochs}")
    print(f"TOTAL = {total_max}")
    for idx, row in exp_max_epochs.iterrows():
        print(f"  {row['experiment']}")
    
    # Armazenar dados para arquivo
    analise_geral['analise_por_horizonte'][horizonte] = {
        'menor_epocas': {
            'num_epochs': int(min_epochs),
            'total': total_min,
            'experimentos': exp_min_epochs['experiment'].tolist()
        },
        'maior_epocas': {
            'num_epochs': int(max_epochs),
            'total': total_max,
            'experimentos': exp_max_epochs['experiment'].tolist()
        }
    }

# Salvar análise geral
with open(f"{OUTPUT_ANALYSIS_DIR}/analise_geral.json", 'w', encoding='utf-8') as f:
    json.dump(analise_geral, f, indent=2, ensure_ascii=False)

# ========================
# PARTE 2: CARREGAR df_global COM RANKING
# ========================

# IMPORTANTE: df_global deve ser passado como variável ou carregado de um arquivo
# Aqui vou assumir que você vai passar o df_global ou ele está em um arquivo CSV

# Se df_global já existe na memória, use ele diretamente
# Se não, descomente a linha abaixo e ajuste o caminho
# df_global = pd.read_csv("caminho/para/df_global.csv")

# Para este código funcionar, você precisa garantir que df_global existe
# Vou adicionar uma verificação

try:
    # Tenta usar df_global se já estiver na memória
    df_global
except NameError:
    # Se não estiver, tenta carregar de arquivo
    print("\n⚠️ df_global não encontrado na memória, tentando carregar de arquivo...")
    
    # Tenta vários caminhos possíveis
    possible_paths = [
        "df_global.csv",
        f"{BASE_OUTPUT_PATH}/df_global.csv",
        f"{BASE_OUTPUT_PATH}/../df_global.csv",
        "resultados_globais.csv",
        f"{BASE_OUTPUT_PATH}/resultados_globais.csv"
    ]
    
    df_global = None
    for path in possible_paths:
        try:
            df_global = pd.read_csv(path)
            break
        except FileNotFoundError:
            continue
    
    if df_global is None:
        print("\n❌ ERRO: df_global não encontrado!")
        print("Por favor, forneça o arquivo df_global.csv ou defina a variável df_global")
        print("O arquivo deve ter as colunas:")
        print("  - Experimento")
        print("  - Prob_MyMoE_vs_Time-MoE")
        print("  - Prob_MyMoE_vs_Timer")
        print("  - Prob_MyMoE_vs_TimesFM")
        print("  - Prob_MyMoE_vs_Morai")
        print("  - Prob_MyMoE_vs_Chronos")
        print("  - Score_Medio_MyMoE")
        exit(1)

# Verifica colunas necessárias
required_cols = ['Experimento', 'Score_Medio_MyMoE']
missing_cols = [col for col in required_cols if col not in df_global.columns]

if missing_cols:
    print(f"\n❌ ERRO: df_global está faltando as colunas: {missing_cols}")
    print(f"Colunas disponíveis: {df_global.columns.tolist()}")
    exit(1)

# ========================
# TOP EXPERIMENTOS QUE MAIS GANHAM
# ========================


print("\n" + "=" * 80)
print("DETALHAMENTO DOS TOP EXPERIMENTOS QUE MAIS GANHAM")
print("=" * 80)

top_ganham = df_global.head(TOP_K)
analise_top_ganham = []

for idx, (_, row) in enumerate(top_ganham.iterrows(), 1):
    experiment_name = row['Experimento']
    
    print("\n" + "=" * 80)
    print(f"Experimento: {experiment_name} (TOP {idx})")
    print("=" * 80)
    
    # Busca detalhes diretamente na pasta do experimento
    details = get_experiment_details_by_base_name(experiment_name, base_path)
    
    if not details:
        print(f"⚠️ Dados não encontrados para {experiment_name}")
        continue
    
    exp_analysis = {
        'ranking': idx,
        'experimento': experiment_name,
        'score_medio': float(row['Score_Medio_MyMoE']),
        'horizontes': {}
    }
    
    # Adiciona as probabilidades se existirem
    prob_cols = [col for col in df_global.columns if col.startswith('Prob_MyMoE_vs_')]
    for col in prob_cols:
        exp_analysis[col] = float(row[col])
    
    # Mostra detalhes por horizonte
    for horizonte in sorted(details.keys()):
        info = details[horizonte]
        
        print(f"\n{horizonte.upper()}:")
        
        max_loss_str = f"{info['max_last_loss']:.2f}" if info['max_last_loss'] is not None else "N/A"
        min_loss_str = f"{info['min_last_loss']:.2f}" if info['min_last_loss'] is not None else "N/A"
        
        print(f"  Maior número de Épocas: {info['max_epochs']:<3} | Última Loss para essa época: {max_loss_str:>6} | Ano = {info['max_year']}")
        print(f"  Menor número de Épocas: {info['min_epochs']:<3} | Última Loss para essa época: {min_loss_str:>6} | Ano = {info['min_year']}")
        print(f"  Média de épocas de todos os anos = {info['media_epochs']:.1f}")
        
        exp_analysis['horizontes'][horizonte] = {
            'max_epochs': info['max_epochs'],
            'max_last_loss': float(info['max_last_loss']) if info['max_last_loss'] else None,
            'max_year': info['max_year'],
            'min_epochs': info['min_epochs'],
            'min_last_loss': float(info['min_last_loss']) if info['min_last_loss'] else None,
            'min_year': info['min_year'],
            'media_epochs': float(info['media_epochs'])
        }
    
    # Estatísticas de learners
    learner_stats = get_learner_stats_for_base_name(experiment_name, base_path)
    
    if not learner_stats.empty:
        print(f"\nEstatísticas de Learners:")
        learners_list = []
        for _, learner_row in learner_stats.iterrows():
            print(f"  {learner_row['Learner']:15s} | Seleções: {learner_row['Total Seleções']:8d} | Peso médio: {learner_row['Peso Médio']:.4f}")
            learners_list.append({
                'learner': learner_row['Learner'],
                'total_selecoes': int(learner_row['Total Seleções']),
                'peso_medio': float(learner_row['Peso Médio'])
            })
        exp_analysis['learners'] = learners_list
    
    analise_top_ganham.append(exp_analysis)

# Salvar análise dos top que mais ganham
with open(f"{OUTPUT_ANALYSIS_DIR}/top_{TOP_K}_ganham.json", 'w', encoding='utf-8') as f:
    json.dump(analise_top_ganham, f, indent=2, ensure_ascii=False)

# Criar CSV resumido
top_ganham_summary = []
for exp in analise_top_ganham:
    for horizonte, data in exp['horizontes'].items():
        summary_row = {
            'Ranking': exp['ranking'],
            'Experimento': exp['experimento'],
            'Score_Medio_MyMoE': exp['score_medio'],
            'Horizonte': horizonte,
            'Max_Epocas': data['max_epochs'],
            'Max_Ultima_Loss': data['max_last_loss'],
            'Max_Ano': data['max_year'],
            'Min_Epocas': data['min_epochs'],
            'Min_Ultima_Loss': data['min_last_loss'],
            'Min_Ano': data['min_year'],
            'Media_Epocas': data['media_epochs']
        }
        
        # Adiciona probabilidades
        for key in exp.keys():
            if key.startswith('Prob_MyMoE_vs_'):
                summary_row[key] = exp[key]
        
        top_ganham_summary.append(summary_row)

df_top_ganham = pd.DataFrame(top_ganham_summary)
df_top_ganham.to_csv(f"{OUTPUT_ANALYSIS_DIR}/top_{TOP_K}_ganham_resumo.csv", index=False)


# ========================
# TOP EXPERIMENTOS QUE MAIS PERDEM
# ========================

print("\n" + "=" * 80)
print("DETALHAMENTO DOS TOP EXPERIMENTOS QUE MAIS PERDEM")
print("=" * 80)

top_perdem = df_global.tail(TOP_K)
analise_top_perdem = []

for idx, (_, row) in enumerate(top_perdem.iterrows(), 1):
    experiment_name = row['Experimento']
    
    print("\n" + "=" * 80)
    print(f"Experimento: {experiment_name} (PIOR {idx})")
    print("=" * 80)
    
    # Busca detalhes diretamente na pasta do experimento
    details = get_experiment_details_by_base_name(experiment_name, base_path)
    
    if not details:
        print(f"⚠️ Dados não encontrados para {experiment_name}")
        continue
    
    exp_analysis = {
        'ranking': idx,
        'experimento': experiment_name,
        'score_medio': float(row['Score_Medio_MyMoE']),
        'horizontes': {}
    }
    
    # Adiciona as probabilidades se existirem
    prob_cols = [col for col in df_global.columns if col.startswith('Prob_MyMoE_vs_')]
    for col in prob_cols:
        exp_analysis[col] = float(row[col])
    
    # Mostra detalhes por horizonte
    for horizonte in sorted(details.keys()):
        info = details[horizonte]
        
        print(f"\n{horizonte.upper()}:")
        
        max_loss_str = f"{info['max_last_loss']:.2f}" if info['max_last_loss'] is not None else "N/A"
        min_loss_str = f"{info['min_last_loss']:.2f}" if info['min_last_loss'] is not None else "N/A"
        
        print(f"  Maior número de Épocas: {info['max_epochs']:<3} | Última Loss para essa época: {max_loss_str:>6} | Ano = {info['max_year']}")
        print(f"  Menor número de Épocas: {info['min_epochs']:<3} | Última Loss para essa época: {min_loss_str:>6} | Ano = {info['min_year']}")
        print(f"  Média de épocas de todos os anos = {info['media_epochs']:.1f}")
        
        exp_analysis['horizontes'][horizonte] = {
            'max_epochs': info['max_epochs'],
            'max_last_loss': float(info['max_last_loss']) if info['max_last_loss'] else None,
            'max_year': info['max_year'],
            'min_epochs': info['min_epochs'],
            'min_last_loss': float(info['min_last_loss']) if info['min_last_loss'] else None,
            'min_year': info['min_year'],
            'media_epochs': float(info['media_epochs'])
        }
    
    # Estatísticas de learners
    learner_stats = get_learner_stats_for_base_name(experiment_name, base_path)
    
    if not learner_stats.empty:
        print(f"\nEstatísticas de Learners:")
        learners_list = []
        for _, learner_row in learner_stats.iterrows():
            print(f"  {learner_row['Learner']:15s} | Seleções: {learner_row['Total Seleções']:8d} | Peso médio: {learner_row['Peso Médio']:.4f}")
            learners_list.append({
                'learner': learner_row['Learner'],
                'total_selecoes': int(learner_row['Total Seleções']),
                'peso_medio': float(learner_row['Peso Médio'])
            })
        exp_analysis['learners'] = learners_list
    
    analise_top_perdem.append(exp_analysis)

# Salvar análise dos top que mais perdem
with open(f"{OUTPUT_ANALYSIS_DIR}/top_{TOP_K}_perdem.json", 'w', encoding='utf-8') as f:
    json.dump(analise_top_perdem, f, indent=2, ensure_ascii=False)

# Criar CSV resumido
top_perdem_summary = []
for exp in analise_top_perdem:
    for horizonte, data in exp['horizontes'].items():
        summary_row = {
            'Ranking': exp['ranking'],
            'Experimento': exp['experimento'],
            'Score_Medio_MyMoE': exp['score_medio'],
            'Horizonte': horizonte,
            'Max_Epocas': data['max_epochs'],
            'Max_Ultima_Loss': data['max_last_loss'],
            'Max_Ano': data['max_year'],
            'Min_Epocas': data['min_epochs'],
            'Min_Ultima_Loss': data['min_last_loss'],
            'Min_Ano': data['min_year'],
            'Media_Epocas': data['media_epochs']
        }
        
        # Adiciona probabilidades
        for key in exp.keys():
            if key.startswith('Prob_MyMoE_vs_'):
                summary_row[key] = exp[key]
        
        top_perdem_summary.append(summary_row)

df_top_perdem = pd.DataFrame(top_perdem_summary)
df_top_perdem.to_csv(f"{OUTPUT_ANALYSIS_DIR}/top_{TOP_K}_perdem_resumo.csv", index=False)

print("\n" + "=" * 80)
print("✅ Análise completa!")
print("=" * 80)
print(f"\n📁 Arquivos salvos em: {OUTPUT_ANALYSIS_DIR}/")
print(f"   - analise_geral.json")
print(f"   - top_{TOP_K}_ganham.json")
print(f"   - top_{TOP_K}_ganham_resumo.csv")
print(f"   - top_{TOP_K}_perdem.json")
print(f"   - top_{TOP_K}_perdem_resumo.csv")